In [1]:
import os
import random
import warnings
import math
import time
from pathlib import Path
from PIL import Image
import cv2
import numpy as np
from joblib import Parallel, delayed, cpu_count
from deap import base, creator, tools, gp
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy.ndimage import gaussian_filter, sobel

# ==========================================
# CẤU HÌNH HỆ THỐNG VÀ MÔI TRƯỜNG
# ==========================================
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

# Kiểm tra RAPIDS cuML để tăng tốc Surrogate (Random Forest)
try:
    import cupy as cp
    from cuml.ensemble import RandomForestRegressor as RF_GPU
    print("[HỆ THỐNG] Đã tìm thấy RAPIDS cuML. Surrogate dùng GPU, SVM dùng CPU.")
except ImportError:
    from sklearn.ensemble import RandomForestRegressor as RF_GPU
    print("[HỆ THỐNG] Không có cuML. Chạy toàn bộ bằng CPU.")

LIMIT = 1e9 
NUM_TREES = 15  
POP_SIZE = 100
MAX_GEN = 50
CXPB = 0.8   
MUTPB = 0.19 
ELITISM = int(0.01 * POP_SIZE) 
P_SIZE = int(0.2 * POP_SIZE) 

# --- CẤU HÌNH CHO TẬP WEBCAM ---
THU_MUC_GOC = '/kaggle/input/datasets/xixuhu/office31/Office-31/webcam'
THU_MUC_MOI = '/kaggle/working/webcam_split_50x50'
DATA_DIR_TRAIN = os.path.join(THU_MUC_MOI, 'train')
DATA_DIR_TEST = os.path.join(THU_MUC_MOI, 'test')

# ==========================================
# 1. HÀM CHUẨN BỊ VÀ CHIA TẬP DATASET
# ==========================================
def prepare_train_test_dataset(src_dir, dest_dir, num_train=5, img_size=(50, 50)):
    duong_dan_goc = Path(src_dir)
    thu_muc_train = Path(dest_dir) / 'train'
    thu_muc_test = Path(dest_dir) / 'test'
    
    thu_muc_train.mkdir(parents=True, exist_ok=True)
    thu_muc_test.mkdir(parents=True, exist_ok=True)
    
    dinh_dang_anh = {'.jpg', '.jpeg', '.png', '.bmp'}
    tong_train = tong_test = 0
    
    print(f"Bắt đầu chia tập Train/Test (Train: {num_train} ảnh/class, Resize: {img_size})...", flush=True)
    
    for thu_muc_con in duong_dan_goc.iterdir():
        if not thu_muc_con.is_dir(): continue
            
        (thu_muc_train / thu_muc_con.name).mkdir(parents=True, exist_ok=True)
        (thu_muc_test / thu_muc_con.name).mkdir(parents=True, exist_ok=True)
        
        danh_sach_anh = [f for f in thu_muc_con.glob('*') if f.is_file() and f.suffix.lower() in dinh_dang_anh]
        random.shuffle(danh_sach_anh)
        
        so_luong_train = min(num_train, len(danh_sach_anh))
        anh_train = danh_sach_anh[:so_luong_train]
        anh_test = danh_sach_anh[so_luong_train:]
        
        def process_and_save(img_paths, dest_folder):
            count = 0
            for path in img_paths:
                try:
                    with Image.open(path) as img:
                        img = img.convert('RGB')
                        img_resized = img.resize(img_size, Image.Resampling.LANCZOS)
                        img_resized.save(dest_folder / path.name)
                        count += 1
                except Exception:
                    pass
            return count

        tong_train += process_and_save(anh_train, thu_muc_train / thu_muc_con.name)
        tong_test += process_and_save(anh_test, thu_muc_test / thu_muc_con.name)
        
    print(f"Hoàn thành! Tổng Train: {tong_train} ảnh | Tổng Test: {tong_test} ảnh\n" + "-"*50)

# ==========================================
# 2. HÀM TẢI ẢNH VÀO RAM
# ==========================================
def load_amazon_dataset_raw(data_dir):
    images, labels = [], []
    if not os.path.exists(data_dir):
        return np.array([]), np.array([])

    class_names = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    class_mapping = {name: idx for idx, name in enumerate(class_names)}
    
    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        for img_name in os.listdir(class_dir):
            img = cv2.imread(os.path.join(class_dir, img_name))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img / 255.0) 
                labels.append(class_mapping[class_name])
                
    return np.array(images), np.array(labels, dtype=np.int32)

# ==========================================
# 3. CÁC TOÁN TỬ XỬ LÝ ẢNH MÀU 3D VÀ KHỞI TẠO GP
# ==========================================
def img_add(img1, img2): return np.clip(img1 + img2, 0.0, 1.0)
def img_sub(img1, img2): return np.clip(img1 - img2, 0.0, 1.0)
def img_max(img1, img2): return np.maximum(img1, img2)
def img_relu(img): return np.maximum(0, img)
def img_gaussian_blur(img):
    result = np.zeros_like(img)
    for c in range(3): result[:, :, c] = gaussian_filter(img[:, :, c], sigma=1.0)
    return result
def img_sobel_edges(img):
    result = np.zeros_like(img)
    for c in range(3):
        sx = sobel(img[:, :, c], axis=0)
        sy = sobel(img[:, :, c], axis=1)
        result[:, :, c] = np.hypot(sx, sy)
    return np.clip(result, 0.0, 1.0)

if "FitnessMax" not in creator.__dict__:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    # Thêm fold_accuracies để chạy Lexicase Selection
    creator.create("Individual", list, fitness=creator.FitnessMax, real_fitness=None, parents_fitness=list, hash_str=None, fold_accuracies=list)

pset = gp.PrimitiveSet("MAIN", 1)
pset.renameArguments(ARG0="Image")
pset.addPrimitive(img_add, 2); pset.addPrimitive(img_sub, 2); pset.addPrimitive(img_max, 2)
pset.addPrimitive(img_gaussian_blur, 1); pset.addPrimitive(img_sobel_edges, 1); pset.addPrimitive(img_relu, 1)

toolbox = base.Toolbox()
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=3)
toolbox.register("tree", tools.initIterate, gp.PrimitiveTree, toolbox.expr)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.tree, n=NUM_TREES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# ==========================================
# 4. HÀM ĐÁNH GIÁ (EVALUATION & TRANSFORM)
# ==========================================
def gp_transform_data(individual, X_data, pset_local):
    funcs = [gp.compile(expr=tree, pset=pset_local) for tree in individual]
    num_features = NUM_TREES * 3
    X_features = np.zeros((X_data.shape[0], num_features))
    
    for i in range(X_data.shape[0]):
        img_input = X_data[i]
        for j, func in enumerate(funcs):
            col_idx = j * 3
            try:
                processed_img = func(img_input)
                feature_values = np.mean(processed_img, axis=(0, 1))
                feature_values = np.nan_to_num(feature_values, nan=0.0, posinf=1.0, neginf=0.0)
                X_features[i, col_idx : col_idx+3] = feature_values
            except Exception:
                X_features[i, col_idx : col_idx+3] = 0.0
    return X_features

def evaluate_real_fitness(X_features, y_data):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accuracies = []
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        X_features = np.nan_to_num(X_features, nan=0.0, posinf=LIMIT, neginf=-LIMIT)
    
    X_features_calc = np.array(X_features, dtype=np.float32)
    y_calc = np.array(y_data, dtype=np.float32)
        
    for train_index, test_index in skf.split(X_features_calc, y_calc):
        clf = SVC(kernel='linear', max_iter=2000, random_state=42) 
        X_train, X_test = X_features_calc[train_index], X_features_calc[test_index]
        y_train, y_test = y_calc[train_index], y_calc[test_index]
        
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                scaler = StandardScaler()
                X_train = scaler.fit_transform(X_train)
                X_test = scaler.transform(X_test)
                
                clf.fit(X_train, y_train)
                preds = clf.predict(X_test)
                acc = float(np.mean(preds == y_test))
        except Exception: 
            acc = 0.0
        accuracies.append(acc)
        
    return float(np.mean(accuracies)), accuracies

def evaluate_individual_wrapper(args):
    warnings.filterwarnings('ignore')
    ind_hash, ind, X_data, y_data, pset_local = args
    X_feats = gp_transform_data(ind, X_data, pset_local)
    real_acc, fold_accs = evaluate_real_fitness(X_feats, y_data)
    return (ind_hash, real_acc, fold_accs)

# --- KHAI BÁO CÁC TOÁN TỬ TIẾN HÓA VÀ CHỌN LỌC ---
toolbox.register("select_tournament", tools.selTournament, tournsize=5)

def select_lexicase(individuals, k):
    selected = []
    for _ in range(k):
        candidates = individuals[:]
        cases = list(range(len(candidates[0].fold_accuracies)))
        random.shuffle(cases) 
        
        for case in cases:
            max_val = max(ind.fold_accuracies[case] for ind in candidates)
            candidates = [ind for ind in candidates if ind.fold_accuracies[case] == max_val]
            if len(candidates) == 1:
                break
                
        selected.append(random.choice(candidates))
    return selected

def cxMultiTree(ind1, ind2):
    idx = random.randint(0, NUM_TREES-1)
    ind1[idx], ind2[idx] = gp.cxOnePoint(ind1[idx], ind2[idx])
    return ind1, ind2

def mutMultiTree(individual):
    idx = random.randint(0, NUM_TREES-1)
    mutant = gp.mutUniform(individual[idx], toolbox.expr, pset)
    individual[idx] = mutant[0]
    return individual,

toolbox.register("mate", cxMultiTree)
toolbox.register("mutate", mutMultiTree)

# ==========================================
# 5. MÔ HÌNH ENSEMBLE SURROGATE
# ==========================================
def extract_surrogate_features(individual, subset_X, pset):
    tree_feats = []
    for tree in individual:
        tree_feats.extend([len(tree), tree.height]) 
        
    parent_fit = individual.parents_fitness if hasattr(individual, 'parents_fitness') else [0.0, 0.0, 0.0]
    
    class_labels_raw = gp_transform_data(individual, subset_X, pset)
    class_labels = class_labels_raw.flatten() 
        
    return np.array(tree_feats, dtype=np.float32), np.array(parent_fit, dtype=np.float32), np.array(class_labels, dtype=np.float32)

def predict_surrogate(Archive, offspring, E, E_max, subset_X, pset):
    if len(Archive) < 10: return [0.0] * len(offspring)
    
    T_train, P_train, L_train, Y_train = [], [], [], []
    for ind in Archive:
        t, p, l = extract_surrogate_features(ind, subset_X, pset)
        T_train.append(t); P_train.append(p); L_train.append(l)
        Y_train.append(ind.real_fitness)
        
    T_train = np.array(T_train, dtype=np.float32)
    P_train = np.array(P_train, dtype=np.float32)
    L_train = np.array(L_train, dtype=np.float32)
    Y_train = np.array(Y_train, dtype=np.float32)
    All_train = np.concatenate((T_train, P_train, L_train), axis=1)

    n_samples = len(All_train)
    n_bins_dynamic = min(128, n_samples) 

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rf_g = RF_GPU(n_estimators=100, max_depth=14, n_bins=n_bins_dynamic).fit(All_train, Y_train)
        rf_t = RF_GPU(n_estimators=100, max_depth=14, n_bins=n_bins_dynamic).fit(T_train, Y_train)
        rf_p = RF_GPU(n_estimators=100, max_depth=14, n_bins=n_bins_dynamic).fit(P_train, Y_train)
        rf_l = RF_GPU(n_estimators=100, max_depth=14, n_bins=n_bins_dynamic).fit(L_train, Y_train)
    
    w = min(E / E_max, 1.0)
    preds = []
    
    for child in offspring:
        t, p, l = extract_surrogate_features(child, subset_X, pset)
        f_all = np.concatenate((t, p, l)).reshape(1, -1).astype(np.float32)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            yg = rf_g.predict(f_all)[0]
            y1 = rf_t.predict(t.reshape(1, -1).astype(np.float32))[0]
            y2 = rf_p.predict(p.reshape(1, -1).astype(np.float32))[0]
            y3 = rf_l.predict(l.reshape(1, -1).astype(np.float32))[0]
        
        try:
            log_yg = math.log(max(float(yg), 1e-5))
            log_locals = (math.log(max(float(y1), 1e-5)) + math.log(max(float(y2), 1e-5)) + math.log(max(float(y3), 1e-5))) / 3.0
            y_e = w * log_yg + (1 - w) * log_locals
            preds.append(math.exp(min(y_e, 700)))
        except: 
            preds.append(0.0)
            
    return preds

# ==========================================
# 6. VÒNG LẶP TIẾN HÓA VÀ KIỂM ĐỊNH (MAIN)
# ==========================================
def main(X_train_raw, y_train_raw, X_test_raw, y_test_raw):
    # --- BẮT ĐẦU BẤM GIỜ HÀM MAIN ---
    start_time_main = time.time()
    
    n_jobs = cpu_count()
    print(f"\n[HỆ THỐNG] Đang chạy ES-MTGPFL (Ảnh thô, Linear SVM, ENSEMBLE SURROGATE + LEXICASE)")
    print(f"[HỆ THỐNG] Kích hoạt chế độ đa luồng với {n_jobs} nhân CPU...\n", flush=True)

    random_indices = np.random.choice(len(X_train_raw), min(10, len(X_train_raw)), replace=False)
    SUBSET_X = X_train_raw[random_indices]

    E_max = POP_SIZE + (MAX_GEN - 1) * P_SIZE 
    pop = toolbox.population(n=POP_SIZE)
    Archive = []
    Archive_Hashes = set() 
    Evaluation_Table = {}  
    best_acc_global = 0.0
    E_count = 0
    
    print(f"Tổng Evaluate thật tối đa: {E_max} | Kích thước quần thể: {POP_SIZE}")
    print(f"--- Thế hệ 0 (Đánh giá toàn bộ) ---")
    
    tasks = []
    for ind in pop:
        ind.hash_str = str(ind)
        ind.parents_fitness = [0.0, 0.0, 0.0] 
        tasks.append((ind.hash_str, ind, X_train_raw, y_train_raw, pset))
    
    results = Parallel(n_jobs=n_jobs)(delayed(evaluate_individual_wrapper)(task) for task in tasks)
    
    for ind, (hash_str, fit, fold_accs) in zip(pop, results):
        ind.real_fitness = fit
        ind.fitness.values = (fit,)
        ind.fold_accuracies = fold_accs
        Evaluation_Table[hash_str] = (fit, fold_accs)
        E_count += 1
        
        if hash_str not in Archive_Hashes:
            Archive.append(ind)
            Archive_Hashes.add(hash_str)
            
        if fit > best_acc_global: best_acc_global = fit

    for gen in range(1, MAX_GEN + 1):
        print(f"-- Thế hệ {gen}/{MAX_GEN} --", flush=True)
        
        elites = tools.selBest(pop, 1) 
        num_select = len(pop) - 1

        # --- ÁP DỤNG CHIẾN LƯỢC CHỌN LỌC ĐỘNG ---
        if gen <= (MAX_GEN // 2):
            offspring = toolbox.select_tournament(pop, num_select)
            if gen == 1: print("   -> Đang sử dụng Tournament Selection")
        else:
            offspring = select_lexicase(pop, num_select)
            if gen == (MAX_GEN // 2) + 1: print("   -> Đã chuyển sang Lexicase Selection")

        offspring = list(map(toolbox.clone, offspring))

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                p_fit = [child1.real_fitness, child2.real_fitness, (child1.real_fitness+child2.real_fitness)/2]
                toolbox.mate(child1, child2)
                child1.parents_fitness, child2.parents_fitness = p_fit, p_fit
                del child1.fitness.values, child2.fitness.values
                child1.hash_str, child2.hash_str = str(child1), str(child2) 

        for mutant in offspring:
            if random.random() < MUTPB:
                mutant.parents_fitness = [mutant.real_fitness]*3
                toolbox.mutate(mutant)
                del mutant.fitness.values
                mutant.hash_str = str(mutant) 
                
        unevaluated_offspring = []
        for child in offspring:
            if child.hash_str in Evaluation_Table:
                fit, fold_accs = Evaluation_Table[child.hash_str]
                child.real_fitness = fit
                child.fitness.values = (fit,)
                child.fold_accuracies = fold_accs
            else:
                unevaluated_offspring.append(child)
                
        if len(unevaluated_offspring) > 0:
            preds = predict_surrogate(Archive, unevaluated_offspring, E_count, E_max, SUBSET_X, pset)
            num_to_evaluate = min(P_SIZE, len(unevaluated_offspring))
            top_indices = set(np.argsort(preds)[-num_to_evaluate:])
            
            top_individuals = []
            for i, child in enumerate(unevaluated_offspring):
                if i in top_indices:
                    top_individuals.append(child)
                else:
                    child.real_fitness = preds[i]
                    child.fitness.values = (preds[i],)
                    # Gán fold giả lập bằng đúng predict fitness để Lexicase vẫn chạy được
                    child.fold_accuracies = [preds[i]] * 5
            
            if top_individuals:
                tasks_gen = [(ind.hash_str, ind, X_train_raw, y_train_raw, pset) for ind in top_individuals]
                real_results = Parallel(n_jobs=n_jobs)(delayed(evaluate_individual_wrapper)(task) for task in tasks_gen)
                
                for ind, (hash_str, fit, fold_accs) in zip(top_individuals, real_results):
                    ind.real_fitness = fit
                    ind.fitness.values = (fit,)
                    ind.fold_accuracies = fold_accs
                    Evaluation_Table[hash_str] = (fit, fold_accs)
                    E_count += 1
                    
                    if hash_str not in Archive_Hashes:
                        Archive.append(ind)
                        Archive_Hashes.add(hash_str)
                
        pop[:] = elites + offspring
        
        best_gen_ind = tools.selBest(pop, 1)[0]
        if best_gen_ind.real_fitness > best_acc_global: 
            best_acc_global = best_gen_ind.real_fitness
        
        print(f"   -> Số lần Evaluate thật: {E_count}/{E_max}")
        print(f"   -> Kỷ lục thế hệ hiện tại : {best_gen_ind.real_fitness * 100:.2f}%")
        print(f"   -> KỶ LỤC TOÀN CỤC        : {best_acc_global * 100:.2f}%", flush=True)

    # =============== ĐÁNH GIÁ TRÊN TẬP TEST ===============
    print(f"\n================ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP ================")
    best_ind = tools.selBest(pop, 1)[0]
    
    print("1. Trích xuất đặc trưng trên tập TRAIN bằng cá thể vô địch...")
    X_train_feats = gp_transform_data(best_ind, X_train_raw, pset)
    X_train_feats = np.nan_to_num(X_train_feats, nan=0.0, posinf=LIMIT, neginf=-LIMIT)
    
    print("2. Trích xuất đặc trưng trên tập TEST bằng cá thể vô địch...")
    X_test_feats = gp_transform_data(best_ind, X_test_raw, pset)
    X_test_feats = np.nan_to_num(X_test_feats, nan=0.0, posinf=LIMIT, neginf=-LIMIT)
    
    scaler = StandardScaler()
    X_train_feats = scaler.fit_transform(X_train_feats)
    X_test_feats = scaler.transform(X_test_feats)
    
    print("3. Huấn luyện Linear SVM và Dự đoán...")
    clf = SVC(kernel='linear', max_iter=2000, random_state=42)
    clf.fit(X_train_feats, y_train_raw)
    
    preds = clf.predict(X_test_feats)
    test_acc = np.mean(preds == y_test_raw)
    
    # --- KẾT THÚC BẤM GIỜ HÀM MAIN ---
    end_time_main = time.time()
    elapsed_main = end_time_main - start_time_main
    h_main, rem_main = divmod(elapsed_main, 3600)
    m_main, s_main = divmod(rem_main, 60)

    print("-" * 50)
    print(f"Độ chính xác Validation (Cao nhất) : {best_ind.fitness.values[0] * 100:.2f}%")
    print(f"ĐỘ CHÍNH XÁC TEST (BÁO CÁO CUỐI)   : {test_acc * 100:.2f}%")
    print(f"THỜI GIAN CHẠY THUẬT TOÁN (MAIN)   : {int(h_main):02d}h {int(m_main):02d}m {s_main:05.2f}s")
    print("==================================================================")

# ==========================================
# KHỞI CHẠY CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    # --- BẮT ĐẦU BẤM GIỜ TOÀN BỘ CHƯƠNG TRÌNH ---
    start_time_total = time.time()

    if not os.path.exists(THU_MUC_MOI):
        prepare_train_test_dataset(THU_MUC_GOC, THU_MUC_MOI, num_train=5, img_size=(50, 50))
    else:
        print(f"Thư mục {THU_MUC_MOI} đã tồn tại. Bỏ qua bước tiền xử lý.")

    print("\n--- ĐANG TẢI TẬP TRAIN ---")
    X_train, y_train = load_amazon_dataset_raw(DATA_DIR_TRAIN)
    print(f"Kích thước tập Train: {X_train.shape}")
    
    print("--- ĐANG TẢI TẬP TEST ---")
    X_test, y_test = load_amazon_dataset_raw(DATA_DIR_TEST)
    print(f"Kích thước tập Test: {X_test.shape}")

    if len(X_train) > 0 and len(X_test) > 0:
        main(X_train, y_train, X_test, y_test)
    else:
        print("Lỗi: Không tìm thấy dữ liệu!")
        
    # --- KẾT THÚC BẤM GIỜ TOÀN BỘ CHƯƠNG TRÌNH ---
    end_time_total = time.time()
    elapsed_total = end_time_total - start_time_total
    h_total, rem_total = divmod(elapsed_total, 3600)
    m_total, s_total = divmod(rem_total, 60)
    
    print(f"\n[THỐNG KÊ] Tổng thời gian thực thi toàn bộ script: {int(h_total):02d}h {int(m_total):02d}m {s_total:05.2f}s")

[HỆ THỐNG] Đã tìm thấy RAPIDS cuML. Surrogate dùng GPU, SVM dùng CPU.
Bắt đầu chia tập Train/Test (Train: 5 ảnh/class, Resize: (50, 50))...
Hoàn thành! Tổng Train: 155 ảnh | Tổng Test: 640 ảnh
--------------------------------------------------

--- ĐANG TẢI TẬP TRAIN ---
Kích thước tập Train: (155, 50, 50, 3)
--- ĐANG TẢI TẬP TEST ---
Kích thước tập Test: (640, 50, 50, 3)

[HỆ THỐNG] Đang chạy ES-MTGPFL (Ảnh thô, Linear SVM, ENSEMBLE SURROGATE + LEXICASE)
[HỆ THỐNG] Kích hoạt chế độ đa luồng với 4 nhân CPU...

Tổng Evaluate thật tối đa: 1080 | Kích thước quần thể: 100
--- Thế hệ 0 (Đánh giá toàn bộ) ---
-- Thế hệ 1/50 --
   -> Đang sử dụng Tournament Selection
   -> Số lần Evaluate thật: 120/1080
   -> Kỷ lục thế hệ hiện tại : 51.61%
   -> KỶ LỤC TOÀN CỤC        : 51.61%
-- Thế hệ 2/50 --
   -> Số lần Evaluate thật: 140/1080
   -> Kỷ lục thế hệ hiện tại : 53.55%
   -> KỶ LỤC TOÀN CỤC        : 53.55%
-- Thế hệ 3/50 --
   -> Số lần Evaluate thật: 160/1080
   -> Kỷ lục thế hệ hiện tại : 5